# Dam-Explained Irrigation Classification

## Purpose

Classify each CPIS by whether its location is plausibly explained by large-dam access. This directly supports the paper question: has irrigation expansion occurred inside areas supported by large-scale dam infrastructure, or has it shifted toward more decentralized systems?

This notebook intentionally does not use Isolation Forest as the main result. The classification is directional and interpretable:

- `dam_accessible_proxy`: CPIS is within the distance threshold and downslope from the nearest dam.
- `near_dam_but_uphill`: CPIS is near a dam, but gravity-fed access is not elevation-feasible.
- `distant_from_large_dam`: CPIS is beyond the large-dam distance threshold.

Groundwater productivity and NDWI activity are joined after classification as context, not as criteria for whether a CPIS is dam-explained.

## Inputs and Outputs

| Dataset | Config key | Role |
|---|---|---|
| CPIS polygons | `SSA_Combined_CPIS_All_shp_path` | Base CPIS inventory |
| Dam-CPIS profiles | `Dam_CPIS_Profiles_csv_path` | Distance, elevation difference, accessibility class |
| NDWI stats | `CPIS_NDWI_stats_csv_path` | Activity context |
| Groundwater productivity | `CPIS_GP_Groundwater_csv_path` | Groundwater context |
| Classification CSV | `CPIS_Dam_Explained_csv_path` | Tabular output |
| Classification shapefile | `CPIS_Dam_Explained_shp_path` | Spatial output |


In [ ]:
# --- Imports and config ---
import os
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from Code.utils.utility import load_config, resolve_path

config = load_config()
DISTANCE_THRESHOLD_KM = 50


## Load CPIS and Dam-Access Features

The main classification uses only dam-access evidence: distance to nearest dam and elevation feasibility.

In [ ]:
# --- Load CPIS base geometry ---
cpis = gpd.read_file(resolve_path(config['SSA_Combined_CPIS_All_shp_path']))
cpis = cpis.to_crs('EPSG:3857')
cpis['cpis_idx'] = cpis.index
cpis['cpis_area_m2'] = cpis.geometry.area

print(f'CPIS loaded: {len(cpis):,}')
display(cpis.drop(columns='geometry', errors='ignore').head())


In [ ]:
# --- Join dam distance and elevation-feasibility profiles ---
profiles_path = resolve_path(config['Dam_CPIS_Profiles_csv_path'])
if not os.path.isfile(profiles_path):
    raise FileNotFoundError(
        f'Missing {profiles_path}. Run 4_dem_flow_analysis.ipynb first.'
    )

profiles = pd.read_csv(profiles_path, index_col=0)
needed = ['nearest_dam_dist_m', 'elev_m', 'nearest_dam_elev_m', 'elev_diff_m']
available = [c for c in needed + ['accessibility'] if c in profiles.columns]
cpis = cpis.drop(columns=[c for c in available if c in cpis.columns], errors='ignore')
cpis = cpis.join(profiles[available], how='left')

cpis['dist_dam_km'] = cpis['nearest_dam_dist_m'] / 1000

if 'accessibility' not in cpis.columns:
    cpis['accessibility'] = np.select(
        [
            cpis['dist_dam_km'] > DISTANCE_THRESHOLD_KM,
            cpis['elev_diff_m'] < 0,
        ],
        ['infeasible_distance', 'infeasible_elevation'],
        default='feasible',
    )

print('Dam-access classes from profile data:')
print(cpis['accessibility'].value_counts(dropna=False))
display(cpis[['cpis_idx', 'dist_dam_km', 'elev_diff_m', 'accessibility']].head())


## Classify Dam-Explainedness

`least_dam_explained` means a CPIS is either too distant from a large dam or not elevation-feasible for gravity-fed access from the nearest dam. It does not assert the true water source.

In [ ]:
# --- Directional dam-explained classification ---
access = cpis['accessibility'].fillna('unknown').astype(str).str.lower()

cpis['dam_explained_class'] = np.select(
    [
        access.eq('feasible'),
        access.eq('infeasible_elevation'),
        access.eq('infeasible_distance') | (cpis['dist_dam_km'] > DISTANCE_THRESHOLD_KM),
    ],
    [
        'dam_accessible_proxy',
        'near_dam_but_uphill',
        'distant_from_large_dam',
    ],
    default='unknown',
)

cpis['least_dam_explained'] = cpis['dam_explained_class'].isin([
    'near_dam_but_uphill',
    'distant_from_large_dam',
])
cpis['dam_explained_binary'] = np.where(
    cpis['least_dam_explained'],
    'least_dam_explained',
    np.where(cpis['dam_explained_class'].eq('dam_accessible_proxy'), 'dam_accessible_proxy', 'unknown'),
)

class_order = ['dam_accessible_proxy', 'near_dam_but_uphill', 'distant_from_large_dam', 'unknown']
summary = cpis['dam_explained_class'].value_counts().reindex(class_order).fillna(0).astype(int)
summary_pct = (summary / summary.sum() * 100).round(1)
classification_summary = pd.DataFrame({'n_cpis': summary, 'pct_cpis': summary_pct})
display(classification_summary)


## Join Activity and Groundwater Context

These variables help interpret the least-dam-explained CPIS. They are not used to decide whether a CPIS is explained by dams.

In [ ]:
# --- NDWI activity context ---
ndwi_path = resolve_path(config['CPIS_NDWI_stats_csv_path'])
if os.path.isfile(ndwi_path):
    ndwi = pd.read_csv(ndwi_path)
    if 'cpis_idx' in ndwi.columns:
        ndwi = ndwi.set_index('cpis_idx')
    ndwi_cols = [c for c in ['ndvi_mean', 'ndvi_median', 'ndvi_std', 'n_pixels'] if c in ndwi.columns]
    cpis = cpis.join(ndwi[ndwi_cols], how='left')
    cpis['activity_status'] = np.where(
        cpis['ndvi_mean'].isna(),
        'unknown',
        np.where(cpis['ndvi_mean'] > 0, 'active', 'inactive'),
    )
    print(f'Joined NDWI activity context for {cpis["ndvi_mean"].notna().sum():,} CPIS')
else:
    cpis['activity_status'] = 'unknown'
    print(f'NDWI context not found: {ndwi_path}')

# --- Groundwater productivity context ---
gw_path = resolve_path(config['CPIS_GP_Groundwater_csv_path'])
if os.path.isfile(gw_path):
    gw = pd.read_csv(gw_path, index_col=0)
    if 'source_gw_productivity' not in gw.columns and 'gp_gw_productivity' in gw.columns:
        gw['source_gw_productivity'] = gw['gp_gw_productivity']
    gw_cols = [c for c in ['source_gw_productivity', 'productivity_class', 'yield_range'] if c in gw.columns]
    cpis = cpis.join(gw[gw_cols], how='left')
    cpis['groundwater_context'] = np.select(
        [
            cpis['source_gw_productivity'].isna(),
            cpis['source_gw_productivity'] <= 0.75,
            cpis['source_gw_productivity'] > 0.75,
        ],
        ['unknown', 'low_productivity', 'moderate_high_productivity'],
        default='unknown',
    )
    print(f'Joined groundwater context for {cpis["source_gw_productivity"].notna().sum():,} CPIS')
else:
    cpis['groundwater_context'] = 'unknown'
    print(f'Groundwater context not found: {gw_path}')

display(pd.crosstab(cpis['dam_explained_binary'], cpis['activity_status'], margins=True))
display(pd.crosstab(cpis['dam_explained_binary'], cpis['groundwater_context'], margins=True))


## Growth Framing

If a `Year` field is present, summarize the distribution of CPIS by dam-explained class and year. This is the bridge to the inside-vs-outside infrastructure question.

In [ ]:
# --- Year and country summaries ---
if 'Year' in cpis.columns:
    year_summary = pd.crosstab(cpis['Year'], cpis['dam_explained_class'])
    year_pct = (year_summary.div(year_summary.sum(axis=1), axis=0) * 100).round(1)
    print('Counts by year:')
    display(year_summary)
    print('Percent by year:')
    display(year_pct)

country_col = next((c for c in ['Country', 'country', 'ADM0_NAME'] if c in cpis.columns), None)
if country_col:
    country_summary = pd.crosstab(cpis[country_col], cpis['dam_explained_binary'])
    country_summary['pct_least_dam_explained'] = (
        country_summary.get('least_dam_explained', 0) / country_summary.sum(axis=1) * 100
    ).round(1)
    display(country_summary.sort_values('pct_least_dam_explained', ascending=False).head(20))


## Save Outputs

In [ ]:
# --- Save classification outputs ---
csv_path = resolve_path(config.get(
    'CPIS_Dam_Explained_csv_path',
    'Data/Processed/CPIS_Dam_Explained_Classification.csv',
))
shp_path = resolve_path(config.get(
    'CPIS_Dam_Explained_shp_path',
    'Data/Processed/CPIS_Dam_Explained-shp/CPIS_Dam_Explained.shp',
))

os.makedirs(os.path.dirname(csv_path), exist_ok=True)
os.makedirs(os.path.dirname(shp_path), exist_ok=True)

keep_cols = [
    'cpis_idx', 'Year', 'Country', 'Country.Co', 'cpis_area_m2',
    'dist_dam_km', 'nearest_dam_dist_m', 'elev_m', 'nearest_dam_elev_m',
    'elev_diff_m', 'accessibility', 'dam_explained_class',
    'dam_explained_binary', 'least_dam_explained', 'activity_status',
    'ndvi_mean', 'ndvi_median', 'source_gw_productivity',
    'productivity_class', 'yield_range', 'groundwater_context',
]
keep_cols = [c for c in keep_cols if c in cpis.columns]

cpis[keep_cols].to_csv(csv_path, index=False)
cpis.drop(columns=['centroid_x', 'centroid_y'], errors='ignore').to_file(shp_path)

print(f'Classification CSV saved: {csv_path}')
print(f'Classification shapefile saved: {shp_path}')


## Quick-Look Maps and Distributions

In [ ]:
# --- Plot map and class distribution ---
palette = {
    'dam_accessible_proxy': '#1a9850',
    'near_dam_but_uphill': '#f46d43',
    'distant_from_large_dam': '#d73027',
    'unknown': '#969696',
}

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

classification_summary['n_cpis'].plot(
    kind='barh',
    ax=axes[0],
    color=[palette.get(k, '#969696') for k in classification_summary.index],
)
axes[0].set_xlabel('CPIS count')
axes[0].set_ylabel('Dam-explained class')
axes[0].set_title('CPIS by Dam-Explained Class')

for cls, sub in cpis.groupby('dam_explained_class'):
    sub.plot(ax=axes[1], color=palette.get(cls, '#969696'), markersize=2, alpha=0.7, label=cls)
axes[1].set_title('Spatial Distribution')
axes[1].set_axis_off()
axes[1].legend(loc='lower left', markerscale=4)

plt.tight_layout()
plt.show()


## Interpretation Notes

- The main paper result should report the share and growth of CPIS in `dam_accessible_proxy` versus `least_dam_explained` classes.
- `near_dam_but_uphill` is useful because it separates radial proximity from serviceability.
- Groundwater productivity should be reported as context among least-dam-explained CPIS, not as evidence that those CPIS are or are not dam-supported.
- NDWI or a stronger activity classifier should be used to restrict conclusions to actively irrigated CPIS when the question is effective irrigation rather than mapped infrastructure.